In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet").exists():
    PROJECT_ROOT = Path.cwd().parent

IMAGE_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
PRODUCT_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "canonical_products.parquet"

assert IMAGE_DATA_PATH.exists(), f"Image dataset not found: {IMAGE_DATA_PATH}"
assert PRODUCT_DATA_PATH.exists(), f"Product dataset not found: {PRODUCT_DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Image dataset:", IMAGE_DATA_PATH)
print("Product dataset:", PRODUCT_DATA_PATH)

Project root: c:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer
Image dataset: c:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer\data\processed\canonical_images.parquet
Product dataset: c:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer\data\processed\canonical_products.parquet


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate the project root
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "processed").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

IMAGE_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
)
PRODUCT_DATA_PATH = (
    PROJECT_ROOT / "data" / "processed" / "canonical_products.parquet"
)

assert IMAGE_DATA_PATH.exists(), f"Image dataset not found: {IMAGE_DATA_PATH}"
assert PRODUCT_DATA_PATH.exists(), f"Product dataset not found: {PRODUCT_DATA_PATH}"

print("Project root:", PROJECT_ROOT)
print("Image dataset found:", IMAGE_DATA_PATH.exists())
print("Product dataset found:", PRODUCT_DATA_PATH.exists())

Project root: c:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer
Image dataset found: True
Product dataset found: True


In [3]:
images_df = pd.read_parquet(IMAGE_DATA_PATH)
products_df = pd.read_parquet(PRODUCT_DATA_PATH)

print("Image dataset shape:", images_df.shape)
print("Product dataset shape:", products_df.shape)

display(images_df.head())

Image dataset shape: (3909, 15)
Product dataset shape: (27789, 21)


,image_id,canonical_product_id,marketplace,image_url,local_path,image_type,download_status,file_size_bytes,width,height,content_hash_sha256,perceptual_hash_dhash,ocr_text,packaging_consistency_score,source_provenance
0,amazon_B00V5CDD9C_img_0,amazon_B00V5CDD9C,amazon,https://m.media-amazon.com/images/I/3117rKqnXU...,data/raw/images/cache\B00V5CDD9C_img_0.jpg,primary,DOWNLOADED,15560.0,500.0,500.0,29e11f2e380188cebae70d830c5ccc1b96921c7e1dff22...,96969696b2b69600,BEAUTIFY- DUO-PRO HAIR EXTENSION TAPE 4 anxOßcm,0.50,Amazon Reviews'23 (Beauty & Personal Care)
1,amazon_B085B5S92V_img_0,amazon_B085B5S92V,amazon,https://m.media-amazon.com/images/I/41jJEuteHF...,data/raw/images/cache\B085B5S92V_img_0.jpg,primary,DOWNLOADED,23668.0,500.0,500.0,4f9fcd61b350b364309e13398fc2cae0b955787ae5c9e7...,dc19939226643c22,NaN,0.50,Amazon Reviews'23 (Beauty & Personal Care)
2,amazon_B088G1ZJHP_img_0,amazon_B088G1ZJHP,amazon,https://m.media-amazon.com/images/I/41lvMJiwzR...,data/raw/images/cache\B088G1ZJHP_img_0.jpg,primary,DOWNLOADED,30868.0,500.0,500.0,80229026071ee717d7a53b6e81c891b04b82308831639a...,78dbd3ccd1d1d171,I FUEL 4 Neon-Argon Fus&l,0.00,Amazon Reviews'23 (Beauty & Personal Care)
3,amazon_B01M1OFZOG_img_0,amazon_B01M1OFZOG,amazon,https://m.media-amazon.com/images/I/41QkW6wcVD...,data/raw/images/cache\B01M1OFZOG_img_0.jpg,primary,DOWNLOADED,16259.0,455.0,442.0,88334a8fe7d7c2236cad1b3a77a4516df5e20280200b63...,0913264d992366c7,NaN,0.50,Amazon Reviews'23 (Beauty & Personal Care)
4,amazon_B00JBUJ6QO_img_0,amazon_B00JBUJ6QO,amazon,https://m.media-amazon.com/images/I/51wAzWPHPu...,data/raw/images/cache\B00JBUJ6QO_img_0.jpg,primary,DOWNLOADED,37776.0,500.0,500.0,269c720dd32fe05d3e192f759c0ad71ac150bac1ca56c6...,1f9b9b9a9e969e96,bliss &precise Ire r rem,0.25,Amazon Reviews'23 (Beauty & Personal Care)


In [4]:
print("=== IMAGE DATA QUALITY ===")

print("\nDownload status:")
display(images_df["download_status"].value_counts(dropna=False))

print("\nImage type:")
display(images_df["image_type"].value_counts(dropna=False))

print("\nMissing values:")
display(
    images_df.isna().sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nExact duplicate rows:", images_df.duplicated().sum())

print(
    "\nUnique image IDs:",
    images_df["image_id"].nunique()
)

print(
    "Unique product IDs in image dataset:",
    images_df["canonical_product_id"].nunique()
)

=== IMAGE DATA QUALITY ===

Download status:


download_status
PENDING       3389
DOWNLOADED     520
Name: count, dtype: int64


Image type:


image_type
thumb      1235
hi_res     1019
large       935
primary     720
Name: count, dtype: int64


Missing values:


,missing_count
ocr_text,3705
perceptual_hash_dhash,3389
local_path,3389
width,3389
height,3389
file_size_bytes,3389
content_hash_sha256,3389
packaging_consistency_score,3389
image_type,0
download_status,0



Exact duplicate rows: 0

Unique image IDs: 3909
Unique product IDs in image dataset: 720


In [5]:
product_ids = set(products_df["canonical_product_id"].dropna())
image_product_ids = set(images_df["canonical_product_id"].dropna())

orphan_image_products = image_product_ids - product_ids

print("Unique products:", len(product_ids))
print("Unique products represented in image data:", len(image_product_ids))
print("Image product IDs missing from canonical products:",
      len(orphan_image_products))

assert len(orphan_image_products) == 0, (
    "Some image records reference unknown canonical products."
)

Unique products: 27789
Unique products represented in image data: 720
Image product IDs missing from canonical products: 0


In [6]:
from PIL import Image

print("Pillow is installed and ready.")

Pillow is installed and ready.


In [7]:
# Step 5: Validate downloaded image files

from pathlib import Path
from PIL import Image

downloaded_df = images_df[
    images_df["download_status"].eq("DOWNLOADED")
].copy()

validation_results = []

for _, row in downloaded_df.iterrows():
    image_path = PROJECT_ROOT / Path(row["local_path"])

    result = {
        "image_id": row["image_id"],
        "canonical_product_id": row["canonical_product_id"],
        "file_exists": image_path.is_file(),
        "readable": False,
        "actual_width": None,
        "actual_height": None,
        "actual_size_bytes": None,
        "error": None,
    }

    try:
        if not image_path.is_file():
            raise FileNotFoundError(f"Image not found: {image_path}")

        with Image.open(image_path) as img:
            img.verify()

        with Image.open(image_path) as img:
            result["actual_width"] = img.width
            result["actual_height"] = img.height

        result["actual_size_bytes"] = image_path.stat().st_size
        result["readable"] = True

    except Exception as exc:
        result["error"] = str(exc)

    validation_results.append(result)

image_validation_df = pd.DataFrame(validation_results)

print("Downloaded records:", len(downloaded_df))
print("Files found:", int(image_validation_df["file_exists"].sum()))
print("Readable images:", int(image_validation_df["readable"].sum()))
print("Unreadable or missing:",
      int((~image_validation_df["readable"]).sum()))

display(image_validation_df.head())

Downloaded records: 520
Files found: 520
Readable images: 520
Unreadable or missing: 0


,image_id,canonical_product_id,file_exists,readable,actual_width,actual_height,actual_size_bytes,error
0,amazon_B00V5CDD9C_img_0,amazon_B00V5CDD9C,True,True,500,500,15560,None
1,amazon_B085B5S92V_img_0,amazon_B085B5S92V,True,True,500,500,23668,None
2,amazon_B088G1ZJHP_img_0,amazon_B088G1ZJHP,True,True,500,500,30868,None
3,amazon_B01M1OFZOG_img_0,amazon_B01M1OFZOG,True,True,455,442,16259,None
4,amazon_B00JBUJ6QO_img_0,amazon_B00JBUJ6QO,True,True,500,500,37776,None


In [9]:

# Step 6: Compare recorded metadata with actual files

comparison_df = downloaded_df[
    ["image_id", "width", "height", "file_size_bytes"]
].merge(
    image_validation_df[
        [
            "image_id",
            "readable",
            "actual_width",
            "actual_height",
            "actual_size_bytes",
        ]
    ],
    on="image_id",
    how="left",
    validate="one_to_one",
)

# Compare dimensions and sizes only for readable files
readable_df = comparison_df.loc[
    comparison_df["readable"].eq(True)
].copy()

readable_df["dimensions_match"] = (
    readable_df["width"].eq(readable_df["actual_width"])
    & readable_df["height"].eq(readable_df["actual_height"])
)

readable_df["file_size_matches"] = (
    readable_df["file_size_bytes"].eq(
        readable_df["actual_size_bytes"]
    )
)

print("Readable images:", len(readable_df))
print(
    "Dimension mismatches:",
    int((~readable_df["dimensions_match"]).sum()),
)
print(
    "File-size mismatches:",
    int((~readable_df["file_size_matches"]).sum()),
)

display(
    readable_df.loc[
        (~readable_df["dimensions_match"])
        | (~readable_df["file_size_matches"])
    ]
)


Readable images: 520
Dimension mismatches: 0
File-size mismatches: 0


,image_id,width,height,file_size_bytes,readable,actual_width,actual_height,actual_size_bytes,dimensions_match,file_size_matches


In [10]:

# Step 7: Exact duplicate image analysis

hash_df = images_df[
    images_df["content_hash_sha256"].notna()
    & images_df["content_hash_sha256"].astype("string").str.strip().ne("")
].copy()

repeated_hashes_df = hash_df[
    hash_df.duplicated("content_hash_sha256", keep=False)
].sort_values("content_hash_sha256")

print("Images with SHA-256 hashes:", len(hash_df))
print("Distinct SHA-256 hashes:", hash_df["content_hash_sha256"].nunique())
print("Rows in repeated-hash groups:", len(repeated_hashes_df))
print(
    "Number of repeated-hash groups:",
    repeated_hashes_df["content_hash_sha256"].nunique()
)

display(
    repeated_hashes_df[
        [
            "image_id",
            "canonical_product_id",
            "marketplace",
            "image_type",
            "content_hash_sha256",
        ]
    ]
)


Images with SHA-256 hashes: 520
Distinct SHA-256 hashes: 516
Rows in repeated-hash groups: 5
Number of repeated-hash groups: 1


,image_id,canonical_product_id,marketplace,image_type,content_hash_sha256
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,amazon,primary,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,amazon,primary,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,amazon,primary,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,amazon,primary,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,amazon,primary,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...


In [11]:

# Step 8: Classify exact duplicate groups

if repeated_hashes_df.empty:
    print("No repeated SHA-256 hashes found.")
    duplicate_groups_df = pd.DataFrame()
else:
    duplicate_groups_df = (
        repeated_hashes_df.groupby("content_hash_sha256")
        .agg(
            image_count=("image_id", "size"),
            unique_products=("canonical_product_id", "nunique"),
            image_ids=("image_id", list),
            product_ids=("canonical_product_id", list),
        )
        .reset_index()
    )

    duplicate_groups_df["match_type"] = np.where(
        duplicate_groups_df["unique_products"] > 1,
        "CROSS_PRODUCT_MATCH",
        "SAME_PRODUCT_MATCH",
    )

    display(duplicate_groups_df)

    print("\nMatch type summary:")
    display(duplicate_groups_df["match_type"].value_counts())


,content_hash_sha256,image_count,unique_products,image_ids,product_ids,match_type
0,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...,5,5,"[amazon_B072MDJ9N2_img_0, amazon_B08H1Y77B2_im...","[amazon_B072MDJ9N2, amazon_B08H1Y77B2, amazon_...",CROSS_PRODUCT_MATCH



Match type summary:


match_type
CROSS_PRODUCT_MATCH    1
Name: count, dtype: int64

In [12]:

# Step 9: Repeated perceptual hashes

phash_df = images_df[
    images_df["perceptual_hash_dhash"].notna()
    & images_df["perceptual_hash_dhash"].astype("string").str.strip().ne("")
].copy()

repeated_phashes_df = phash_df[
    phash_df.duplicated("perceptual_hash_dhash", keep=False)
].sort_values("perceptual_hash_dhash")

print("Images with perceptual hashes:", len(phash_df))
print("Distinct perceptual hashes:",
      phash_df["perceptual_hash_dhash"].nunique())
print("Rows in repeated-hash groups:", len(repeated_phashes_df))
print("Number of repeated-hash groups:",
      repeated_phashes_df["perceptual_hash_dhash"].nunique())

display(
    repeated_phashes_df[
        [
            "image_id",
            "canonical_product_id",
            "image_type",
            "perceptual_hash_dhash",
        ]
    ]
)


Images with perceptual hashes: 520
Distinct perceptual hashes: 515
Rows in repeated-hash groups: 7
Number of repeated-hash groups: 2


,image_id,canonical_product_id,image_type,perceptual_hash_dhash
32,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,primary,0c4d4d4d4d4d4d4d
91,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,primary,0c4d4d4d4d4d4d4d
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,primary,d86b23031741338e
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,primary,d86b23031741338e
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,primary,d86b23031741338e
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,primary,d86b23031741338e
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,primary,d86b23031741338e


In [13]:

# Step 10: Initial duplicate-analysis summary

summary = {
    "total_image_records": len(images_df),
    "downloaded_records": int(
        images_df["download_status"].eq("DOWNLOADED").sum()
    ),
    "pending_records": int(
        images_df["download_status"].eq("PENDING").sum()
    ),
    "images_with_sha256": int(
        images_df["content_hash_sha256"].notna().sum()
    ),
    "images_with_perceptual_hash": int(
        images_df["perceptual_hash_dhash"].notna().sum()
    ),
    "exact_duplicate_hash_groups": (
        0 if duplicate_groups_df.empty
        else len(duplicate_groups_df)
    ),
    "repeated_perceptual_hash_groups": (
        repeated_phashes_df["perceptual_hash_dhash"].nunique()
    ),
}

display(pd.Series(summary, name="value").to_frame())


,value
total_image_records,3909
downloaded_records,520
pending_records,3389
images_with_sha256,520
images_with_perceptual_hash,520
exact_duplicate_hash_groups,1
repeated_perceptual_hash_groups,2


In [14]:

# Step 11: Inspect exact duplicate candidates in detail

if repeated_hashes_df.empty:
    print("No exact duplicate candidates found.")
else:
    for hash_value, group in repeated_hashes_df.groupby(
        "content_hash_sha256"
    ):
        print("\n" + "=" * 60)
        print("SHA-256:", hash_value)
        print("Number of image records:", len(group))
        print(
            "Unique products:",
            group["canonical_product_id"].nunique()
        )

        display(
            group[
                [
                    "image_id",
                    "canonical_product_id",
                    "marketplace",
                    "image_type",
                    "local_path",
                ]
            ]
        )



SHA-256: fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e1445500c65f6e8f7170b4
Number of image records: 5
Unique products: 5


,image_id,canonical_product_id,marketplace,image_type,local_path
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,amazon,primary,data/raw/images/cache\B072MDJ9N2_img_0.jpg
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,amazon,primary,data/raw/images/cache\B08H1Y77B2_img_0.jpg
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,amazon,primary,data/raw/images/cache\B089QN5VWB_img_0.jpg
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,amazon,primary,data/raw/images/cache\B01BXV8HH2_img_0.jpg
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,amazon,primary,data/raw/images/cache\B0823BQJ5H_img_0.jpg


In [15]:

# Step 12: Inspect repeated perceptual hashes

if repeated_phashes_df.empty:
    print("No repeated perceptual-hash candidates found.")
else:
    for hash_value, group in repeated_phashes_df.groupby(
        "perceptual_hash_dhash"
    ):
        print("\n" + "=" * 60)
        print("Perceptual hash:", hash_value)
        print("Number of image records:", len(group))
        print(
            "Unique products:",
            group["canonical_product_id"].nunique()
        )

        display(
            group[
                [
                    "image_id",
                    "canonical_product_id",
                    "image_type",
                    "local_path",
                ]
            ]
        )



Perceptual hash: 0c4d4d4d4d4d4d4d
Number of image records: 2
Unique products: 2


,image_id,canonical_product_id,image_type,local_path
32,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,primary,data/raw/images/cache\B07QMZS32S_img_0.jpg
91,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,primary,data/raw/images/cache\B0BWZX4TQ1_img_0.jpg



Perceptual hash: d86b23031741338e
Number of image records: 5
Unique products: 5


,image_id,canonical_product_id,image_type,local_path
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,primary,data/raw/images/cache\B072MDJ9N2_img_0.jpg
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,primary,data/raw/images/cache\B08H1Y77B2_img_0.jpg
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,primary,data/raw/images/cache\B089QN5VWB_img_0.jpg
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,primary,data/raw/images/cache\B01BXV8HH2_img_0.jpg
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,primary,data/raw/images/cache\B0823BQJ5H_img_0.jpg


In [16]:

# Step 13: Prepare perceptual hashes for Hamming-distance analysis

def hamming_distance(hash_a, hash_b):
    """Count differing bits between two hexadecimal hashes."""
    a = int(str(hash_a), 16)
    b = int(str(hash_b), 16)
    return (a ^ b).bit_count()


phash_df = images_df[
    images_df["perceptual_hash_dhash"].notna()
    & images_df["perceptual_hash_dhash"].astype("string").str.strip().ne("")
].copy()

print("Images with perceptual hashes:", len(phash_df))
print("Unique perceptual hashes:", phash_df["perceptual_hash_dhash"].nunique())

# Example to verify the function
example_hash = phash_df["perceptual_hash_dhash"].iloc[0]
print("Example hash:", example_hash)
print("Distance from itself:", hamming_distance(example_hash, example_hash))


Images with perceptual hashes: 520
Unique perceptual hashes: 515
Example hash: 96969696b2b69600
Distance from itself: 0


In [17]:

# Step 14: Find near-duplicate candidates among available hashes

records = phash_df[
    ["image_id", "canonical_product_id", "perceptual_hash_dhash"]
].drop_duplicates("image_id").to_dict("records")

near_duplicate_pairs = []

for i in range(len(records)):
    for j in range(i + 1, len(records)):
        a = records[i]
        b = records[j]

        if a["perceptual_hash_dhash"] == b["perceptual_hash_dhash"]:
            continue

        distance = hamming_distance(
            a["perceptual_hash_dhash"],
            b["perceptual_hash_dhash"],
        )

        if distance <= 2:
            near_duplicate_pairs.append({
                "image_id_1": a["image_id"],
                "product_id_1": a["canonical_product_id"],
                "image_id_2": b["image_id"],
                "product_id_2": b["canonical_product_id"],
                "hamming_distance": distance,
                "same_product": (
                    a["canonical_product_id"] == b["canonical_product_id"]
                ),
            })

near_duplicates_df = pd.DataFrame(near_duplicate_pairs)

print("Near-duplicate pairs found:", len(near_duplicates_df))

if near_duplicates_df.empty:
    print("No distinct-hash pairs found at distance <= 2.")
else:
    display(
        near_duplicates_df.sort_values("hamming_distance")
    )


Near-duplicate pairs found: 5


,image_id_1,product_id_1,image_id_2,product_id_2,hamming_distance,same_product
1,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,1,False
4,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,1,False
2,amazon_B00XYMILHU_img_0,amazon_B00XYMILHU,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,2,False
0,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,amazon_B01LR80A6M_img_0,amazon_B01LR80A6M,2,False
3,amazon_B01LR80A6M_img_0,amazon_B01LR80A6M,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,2,False


In [18]:

# Step 16: Final image-validation report

total_images = len(images_df)

downloaded_count = int(
    images_df["download_status"].eq("DOWNLOADED").sum()
)
pending_count = int(
    images_df["download_status"].eq("PENDING").sum()
)

sha256_count = int(
    images_df["content_hash_sha256"].notna().sum()
)
phash_count = int(
    images_df["perceptual_hash_dhash"].notna().sum()
)

report = pd.DataFrame({
    "Metric": [
        "Total image records",
        "Downloaded records",
        "Pending records",
        "Images with SHA-256 hashes",
        "Images with perceptual hashes",
        "Exact duplicate hash groups",
        "Repeated perceptual hash groups",
        "Near-duplicate pairs (Hamming distance <= 2)",
    ],
    "Value": [
        total_images,
        downloaded_count,
        pending_count,
        sha256_count,
        phash_count,
        len(duplicate_groups_df),
        repeated_phashes_df["perceptual_hash_dhash"].nunique(),
        len(near_duplicates_df),
    ],
})

display(report)


,Metric,Value
0,Total image records,3909
1,Downloaded records,520
2,Pending records,3389
3,Images with SHA-256 hashes,520
4,Images with perceptual hashes,520
5,Exact duplicate hash groups,1
6,Repeated perceptual hash groups,2
7,Near-duplicate pairs (Hamming distance <= 2),5


## Computer Vision — Initial Image Validation

### Objective
Establish the quality, availability, and duplicate-image baseline for the marketplace image dataset before implementing more advanced computer-vision features.

### Current dataset
The canonical image dataset contains 3,909 image records. Of these, 520 are marked as downloaded and 3,389 remain pending. The available downloaded records have image fingerprints that can be used for initial duplicate analysis.

### Validation performed
- Inspected image metadata, image types, and download statuses.
- Checked image-to-product mappings against canonical product records.
- Validated local image readability and compared recorded metadata with actual files.
- Investigated repeated SHA-256 and perceptual-hash values.
- Prepared perceptual-hash Hamming-distance analysis for candidate near-duplicate pairs.

### Limitations
The current fingerprint-based analysis covers only records with available hashes. Pending image records are not yet represented in this analysis. Repeated hashes and near-duplicate candidates are not, by themselves, evidence of fraudulent behavior.

### Next steps
1. Review and document the actual duplicate-analysis results.
2. Decide whether to expand image downloads to improve coverage.
3. Develop and validate duplicate-image detection.
4. Investigate OCR and image-text consistency after image coverage and quality are sufficiently established.

In [19]:
# Step 23: Inspect image coverage by status and image type

print("=== STATUS BY IMAGE TYPE ===")
display(pd.crosstab(
    images_df["image_type"],
    images_df["download_status"],
    margins=True
))

print("\n=== SOURCE PROVENANCE ===")
display(images_df["source_provenance"].value_counts(dropna=False))

print("\n=== PRODUCTS WITH IMAGES ===")
print(
    "Unique image records:",
    images_df["image_id"].nunique()
)
print(
    "Unique products represented:",
    images_df["canonical_product_id"].nunique()
)
print(
    "Unique products with at least one downloaded image:",
    images_df.loc[
        images_df["download_status"].eq("DOWNLOADED"),
        "canonical_product_id"
    ].nunique()
)

=== STATUS BY IMAGE TYPE ===


download_status,DOWNLOADED,PENDING,All
image_type,,,
hi_res,0,1019,1019
large,0,935,935
primary,520,200,720
thumb,0,1235,1235
All,520,3389,3909



=== SOURCE PROVENANCE ===


source_provenance
Amazon Reviews'23 Dense Images                3389
Amazon Reviews'23 (Beauty & Personal Care)     100
Amazon Reviews'23 (Electronics)                 90
Amazon Reviews'23 (Clothing & Fashion)          90
Amazon Reviews'23 (Home & Kitchen)              90
Amazon Reviews'23 (Toys & Games)                90
Amazon Reviews'23 (Books & Stationery)          60
Name: count, dtype: int64


=== PRODUCTS WITH IMAGES ===
Unique image records: 3909
Unique products represented: 720
Unique products with at least one downloaded image: 520


In [21]:
from pathlib import Path
import os

print("Current working directory:")
print(Path.cwd())

print("\nCanonical dataset exists:")
print(Path("data/processed/canonical_images.parquet").exists())

print("\nInterim manifest exists:")
print(Path("data/interim/image_manifest.parquet").exists())

Current working directory:
c:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer\notebooks

Canonical dataset exists:
False

Interim manifest exists:
False


In [22]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    r"C:\Users\shruti\Desktop\project\ai-based-marketplace-analyzer"
)

canonical_path = PROJECT_ROOT / "data" / "processed" / "canonical_images.parquet"
interim_path = PROJECT_ROOT / "data" / "interim" / "image_manifest.parquet"

print("Canonical exists:", canonical_path.exists())
print("Interim exists:", interim_path.exists())

Canonical exists: True
Interim exists: True


In [23]:
canonical_df = pd.read_parquet(canonical_path)
interim_df = pd.read_parquet(interim_path)

print("Canonical dataset:", canonical_df.shape)
print("Interim manifest:", interim_df.shape)

Canonical dataset: (3909, 15)
Interim manifest: (520, 22)


In [24]:
print("=== DATASET SIZES ===")
print("Canonical images:", canonical_df.shape)
print("Interim manifest:", interim_df.shape)

print("\n=== INTERIM COLUMNS ===")
print(interim_df.columns.tolist())

print("\n=== DOWNLOAD STATUS ===")
if "download_status" in interim_df.columns:
    display(interim_df["download_status"].value_counts(dropna=False))

print("\n=== IMAGE ID OVERLAP ===")
if "image_id" in canonical_df.columns and "image_id" in interim_df.columns:
    canonical_ids = set(canonical_df["image_id"].dropna())
    interim_ids = set(interim_df["image_id"].dropna())

    print("Canonical unique IDs:", len(canonical_ids))
    print("Interim unique IDs:", len(interim_ids))
    print("Shared IDs:", len(canonical_ids & interim_ids))
    print("Only in interim:", len(interim_ids - canonical_ids))
    print("Only in canonical:", len(canonical_ids - interim_ids))

print("\n=== IMAGE TYPES ===")
if "image_type" in interim_df.columns:
    display(interim_df["image_type"].value_counts(dropna=False))

=== DATASET SIZES ===
Canonical images: (3909, 15)
Interim manifest: (520, 22)

=== INTERIM COLUMNS ===
['image_id', 'canonical_product_id', 'marketplace', 'category', 'title', 'brand', 'image_type', 'image_url', 'local_path', 'http_status', 'download_status', 'file_size_bytes', 'width', 'height', 'image_format', 'content_hash_sha256', 'perceptual_hash_dhash', 'ocr_text', 'has_readable_text', 'packaging_consistency_score', 'error_message', 'source_provenance']

=== DOWNLOAD STATUS ===


download_status
DOWNLOADED    520
Name: count, dtype: int64


=== IMAGE ID OVERLAP ===
Canonical unique IDs: 3909
Interim unique IDs: 520
Shared IDs: 520
Only in interim: 0
Only in canonical: 3389

=== IMAGE TYPES ===


image_type
primary    520
Name: count, dtype: int64

In [25]:
print("=== CANONICAL COLUMNS ===")
print(canonical_df.columns.tolist())

print("\n=== CANONICAL MISSING VALUES ===")
display(canonical_df.isna().sum().sort_values(ascending=False))

print("\n=== INTERIM MISSING VALUES ===")
display(interim_df.isna().sum().sort_values(ascending=False))

print("\n=== MATCH VALIDATION ===")
matched = canonical_df.merge(
    interim_df[[
        "image_id",
        "canonical_product_id",
        "image_url",
        "content_hash_sha256",
        "perceptual_hash_dhash",
        "ocr_text",
        "download_status"
    ]],
    on="image_id",
    how="inner",
    suffixes=("_canonical", "_interim"),
    validate="one_to_one"
)

print("Matched records:", len(matched))
print(
    "Product ID mismatches:",
    (
        matched["canonical_product_id_canonical"]
        != matched["canonical_product_id_interim"]
    ).sum()
)

print(
    "Image URL mismatches:",
    (
        matched["image_url_canonical"].fillna("")
        != matched["image_url_interim"].fillna("")
    ).sum()
)

print("\n=== EXISTING CANONICAL IMAGE ANALYSIS COVERAGE ===")
for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score"
]:
    if col in canonical_df.columns:
        print(f"{col}: {canonical_df[col].notna().sum()} non-missing values")

=== CANONICAL COLUMNS ===
['image_id', 'canonical_product_id', 'marketplace', 'image_url', 'local_path', 'image_type', 'download_status', 'file_size_bytes', 'width', 'height', 'content_hash_sha256', 'perceptual_hash_dhash', 'ocr_text', 'packaging_consistency_score', 'source_provenance']

=== CANONICAL MISSING VALUES ===


ocr_text                       3705
perceptual_hash_dhash          3389
local_path                     3389
width                          3389
height                         3389
file_size_bytes                3389
content_hash_sha256            3389
packaging_consistency_score    3389
image_type                        0
download_status                   0
marketplace                       0
image_id                          0
canonical_product_id              0
image_url                         0
source_provenance                 0
dtype: int64


=== INTERIM MISSING VALUES ===


error_message                  520
ocr_text                       316
marketplace                      0
category                         0
image_id                         0
canonical_product_id             0
brand                            0
title                            0
image_type                       0
image_url                        0
download_status                  0
file_size_bytes                  0
local_path                       0
http_status                      0
height                           0
width                            0
content_hash_sha256              0
image_format                     0
perceptual_hash_dhash            0
has_readable_text                0
packaging_consistency_score      0
source_provenance                0
dtype: int64


=== MATCH VALIDATION ===
Matched records: 520
Product ID mismatches: 0
Image URL mismatches: 0

=== EXISTING CANONICAL IMAGE ANALYSIS COVERAGE ===
content_hash_sha256: 520 non-missing values
perceptual_hash_dhash: 520 non-missing values
ocr_text: 204 non-missing values
packaging_consistency_score: 520 non-missing values


In [26]:
print("=== MATCH VALIDATION ===")

matched = canonical_df.merge(
    interim_df[
        [
            "image_id",
            "canonical_product_id",
            "image_url",
            "content_hash_sha256",
            "perceptual_hash_dhash",
            "ocr_text",
            "download_status",
        ]
    ],
    on="image_id",
    how="inner",
    suffixes=("_canonical", "_interim"),
    validate="one_to_one",
)

print("Matched records:", len(matched))

print(
    "Product ID mismatches:",
    (
        matched["canonical_product_id_canonical"]
        != matched["canonical_product_id_interim"]
    ).sum(),
)

print(
    "Image URL mismatches:",
    (
        matched["image_url_canonical"].fillna("")
        != matched["image_url_interim"].fillna("")
    ).sum(),
)

print("\n=== CANONICAL ANALYSIS COVERAGE ===")

for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
]:
    if col in canonical_df.columns:
        print(
            f"{col}: "
            f"{canonical_df[col].notna().sum()} / {len(canonical_df)} "
            "non-missing"
        )

print("\n=== INTERIM ANALYSIS COVERAGE ===")

for col in [
    "content_hash_sha256",
    "perceptual_hash_dhash",
    "ocr_text",
    "packaging_consistency_score",
]:
    if col in interim_df.columns:
        print(
            f"{col}: "
            f"{interim_df[col].notna().sum()} / {len(interim_df)} "
            "non-missing"
        )

=== MATCH VALIDATION ===
Matched records: 520
Product ID mismatches: 0
Image URL mismatches: 0

=== CANONICAL ANALYSIS COVERAGE ===
content_hash_sha256: 520 / 3909 non-missing
perceptual_hash_dhash: 520 / 3909 non-missing
ocr_text: 204 / 3909 non-missing
packaging_consistency_score: 520 / 3909 non-missing

=== INTERIM ANALYSIS COVERAGE ===
content_hash_sha256: 520 / 520 non-missing
perceptual_hash_dhash: 520 / 520 non-missing
ocr_text: 204 / 520 non-missing
packaging_consistency_score: 520 / 520 non-missing


In [27]:
print("=== OCR COVERAGE ===")

ocr_available = interim_df["ocr_text"].notna() & (
    interim_df["ocr_text"].astype("string").str.strip() != ""
)

print("Total downloaded images:", len(interim_df))
print("Images with usable OCR text:", int(ocr_available.sum()))
print("Images without usable OCR text:", int((~ocr_available).sum()))
print("OCR coverage:", f"{ocr_available.mean() * 100:.2f}%")

print("\n=== PACKAGING CONSISTENCY SCORES ===")

display(
    interim_df["packaging_consistency_score"]
    .describe()
    .to_frame("score")
)

print("\n=== SCORES BY OCR AVAILABILITY ===")

interim_df = interim_df.copy()
interim_df["ocr_available"] = ocr_available

display(
    interim_df.groupby("ocr_available")[
        "packaging_consistency_score"
    ].agg(["count", "mean", "median", "min", "max"])
)

print("\n=== OCR EXAMPLES ===")

display(
    interim_df.loc[
        ocr_available,
        ["image_id", "title", "ocr_text", "packaging_consistency_score"]
    ].head(10)
)

=== OCR COVERAGE ===
Total downloaded images: 520
Images with usable OCR text: 204
Images without usable OCR text: 316
OCR coverage: 39.23%

=== PACKAGING CONSISTENCY SCORES ===


,score
count,520.000000
mean,0.453500
std,0.221178
min,0.000000
25%,0.500000
50%,0.500000
75%,0.500000
max,1.000000



=== SCORES BY OCR AVAILABILITY ===


,count,mean,median,min,max
ocr_available,,,,,
False,316,0.500000,0.50,0.5,0.5
True,204,0.381471,0.38,0.0,1.0



=== OCR EXAMPLES ===


,image_id,title,ocr_text,packaging_consistency_score
0,amazon_B00V5CDD9C_img_0,DUO-PRO From Walker Tape Co. Extension tabs pa...,BEAUTIFY- DUO-PRO HAIR EXTENSION TAPE 4 anxOßcm,0.50
2,amazon_B088G1ZJHP_img_0,NuDerma FUSION Wand 4-Piece Set - Neon & Argon...,I FUEL 4 Neon-Argon Fus&l,0.00
4,amazon_B00JBUJ6QO_img_0,Bliss Fuzz Off Bikini Fresh-scented Hair Remov...,bliss &precise Ire r rem,0.25
6,amazon_B077TBSVK8_img_0,TRUSS Equilibrium Shampoo for Oily Hair with D...,TRUSS,1.00
7,amazon_B01F8AWWAW_img_0,"Dial Moisturizing Body Wash, Silk & Magnolia w...",NEW Djal SILK. MAGNOLIA RESINING PROTEIN NEW' ...,0.30
8,amazon_B08NHTHD2G_img_0,Warren London Dog Nail Polish in A Bottle Prem...,Tarren,0.00
12,amazon_B00SNVYWBQ_img_0,Teatrical Crema Celulas Madre Ultra Aclaradora...,ACVARYADORA regeneraciön*,0.00
13,amazon_B00487JRXC_img_0,Theatrical Makeup Kits - Fair: Light PK-1,Creme Personal Kit,0.00
14,amazon_B00XYMG1HC_img_0,1 oz - Bargz Perfume - Mambo Body Oil For Wome...,TYPE,0.00
15,amazon_B0B6DPNJHQ_img_0,[SKIN&LAB] Barrierderm Milky Serum | Infused w...,SKIN & LAB BARRIERDERM MILKY SERUM,1.00


In [28]:
print("=== PACKAGING SCORE QUALITY CHECK ===")

scores = pd.to_numeric(
    interim_df["packaging_consistency_score"],
    errors="coerce"
)

print("Missing scores:", scores.isna().sum())
print("Scores below 0:", (scores < 0).sum())
print("Scores above 1:", (scores > 1).sum())

print("\n=== SCORE DISTRIBUTION ===")
display(scores.value_counts(dropna=False).sort_index())

print("\n=== OCR AVAILABILITY VS SCORE ===")
ocr_available = (
    interim_df["ocr_text"].fillna("").astype(str).str.strip().ne("")
)

display(
    interim_df.assign(
        ocr_available=ocr_available,
        packaging_score=scores
    )
    .groupby("ocr_available")["packaging_score"]
    .agg(["count", "mean", "median", "std", "min", "max"])
)

print("\n=== CHECK NEUTRAL SCORES WITHOUT OCR ===")
no_ocr = interim_df.loc[~ocr_available].copy()

print("Images without OCR:", len(no_ocr))
print(
    "Images without OCR assigned score 0.50:",
    int(
        pd.to_numeric(
            no_ocr["packaging_consistency_score"],
            errors="coerce"
        ).eq(0.50).sum()
    )
)

=== PACKAGING SCORE QUALITY CHECK ===
Missing scores: 0
Scores below 0: 0
Scores above 1: 0

=== SCORE DISTRIBUTION ===


packaging_consistency_score
0.00     67
0.07      1
0.14      2
0.15      1
0.17      1
0.20      6
0.25      7
0.27      1
0.29      3
0.30      1
0.33     10
0.36      1
0.38      3
0.40      4
0.43      3
0.44      1
0.47      1
0.50    348
0.56      2
0.57      4
0.60      3
0.62      2
0.67      6
0.71      2
0.74      1
0.75     10
0.83      4
0.86      2
1.00     23
Name: count, dtype: int64


=== OCR AVAILABILITY VS SCORE ===


,count,mean,median,std,min,max
ocr_available,,,,,,
False,316,0.500000,0.50,0.000000,0.5,0.5
True,204,0.381471,0.38,0.341308,0.0,1.0



=== CHECK NEUTRAL SCORES WITHOUT OCR ===
Images without OCR: 316
Images without OCR assigned score 0.50: 316


In [29]:
print("=== NEUTRAL SCORE INVESTIGATION ===")

scores = pd.to_numeric(
    interim_df["packaging_consistency_score"],
    errors="coerce"
)

ocr_available = (
    interim_df["ocr_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

neutral_score = scores.eq(0.50)

print("Total images:", len(interim_df))
print("Images scored 0.50:", int(neutral_score.sum()))

print(
    "Score 0.50 WITH OCR:",
    int((neutral_score & ocr_available).sum())
)

print(
    "Score 0.50 WITHOUT OCR:",
    int((neutral_score & ~ocr_available).sum())
)

print(
    "Non-neutral scores WITHOUT OCR:",
    int((~neutral_score & ~ocr_available).sum())
)

print("\n=== SCORE AND OCR CROSS-TABULATION ===")

display(
    pd.crosstab(
        ocr_available,
        neutral_score,
        rownames=["OCR available"],
        colnames=["Score equals 0.50"],
        margins=True
    )
)

print("\n=== EXAMPLES OF NEUTRAL SCORES WITH OCR ===")

display(
    interim_df.loc[
        neutral_score & ocr_available,
        [
            "image_id",
            "title",
            "ocr_text",
            "packaging_consistency_score"
        ]
    ].head(10)
)

=== NEUTRAL SCORE INVESTIGATION ===
Total images: 520
Images scored 0.50: 348
Score 0.50 WITH OCR: 32
Score 0.50 WITHOUT OCR: 316
Non-neutral scores WITHOUT OCR: 0

=== SCORE AND OCR CROSS-TABULATION ===


Score equals 0.50,False,True,All
OCR available,,,
False,0,316,316
True,172,32,204
All,172,348,520



=== EXAMPLES OF NEUTRAL SCORES WITH OCR ===


,image_id,title,ocr_text,packaging_consistency_score
0,amazon_B00V5CDD9C_img_0,DUO-PRO From Walker Tape Co. Extension tabs pa...,BEAUTIFY- DUO-PRO HAIR EXTENSION TAPE 4 anxOßcm,0.5
28,amazon_B01M7QL7WW_img_0,Janeube Formula X for Sephora Delete All 5 Fin...,FORMULA x DELETE ALL S•FINGER NAIL POLSH REMOV...,0.5
29,amazon_B0BYDRR4ZR_img_0,Onesta Hair Care Plant Based Thickening Condit...,ONESTA CONDITIONE8,0.5
40,amazon_B085ZNSK1X_img_0,Beauty Creations Flawless Stay Poreless Primer,VSS STAY,0.5
46,amazon_B00EORAP4G_img_0,Temporary Tattoo Factory Fantasy Lower Back Ta...,"(Fantasy Lower (Back, Tattoos",0.5
47,amazon_B01BMY0F2U_img_0,Rugged Riley All Natural Men's Rustic Woods Be...,RUGGED RILEY lab (30mL),0.5
78,amazon_B0888TZ9W6_img_0,Large Body Wipes for Adults Bathing. Biodegrad...,AV E N - 12'12• BODY WIPES . e 'G CLEAN.,0.5
79,amazon_B0728C52YV_img_0,AICHUN BEAUTY 1pc/50g Body Cream For Armpit El...,on skin For Armpit & Between Legs Cream For Ar...,0.5
93,amazon_B01M8JN3UX_img_0,Becca Ultimate Coverage 24 hour Foundation Mah...,"f,",0.5
99,amazon_B018H2BWZE_img_0,Bath Body Works Aromatherapy Stress Eucalyptus...,stress relief,0.5


In [30]:
print("=== EXACT DUPLICATE SUMMARY ===")

exact_hashes = (
    canonical_df.loc[
        canonical_df["content_hash_sha256"].notna()
        & canonical_df["content_hash_sha256"].ne(""),
        ["image_id", "canonical_product_id", "content_hash_sha256"]
    ]
)

exact_groups = exact_hashes[
    exact_hashes.duplicated("content_hash_sha256", keep=False)
].sort_values("content_hash_sha256")

print("Records with repeated exact hashes:", len(exact_groups))
print(
    "Distinct repeated exact-hash groups:",
    exact_groups["content_hash_sha256"].nunique()
)

display(exact_groups.head(20))

print("\n=== PERCEPTUAL DUPLICATE SUMMARY ===")

perceptual_hashes = (
    canonical_df.loc[
        canonical_df["perceptual_hash_dhash"].notna()
        & canonical_df["perceptual_hash_dhash"].ne(""),
        ["image_id", "canonical_product_id", "perceptual_hash_dhash"]
    ]
)

perceptual_groups = perceptual_hashes[
    perceptual_hashes.duplicated("perceptual_hash_dhash", keep=False)
].sort_values("perceptual_hash_dhash")

print("Records with repeated perceptual hashes:", len(perceptual_groups))
print(
    "Distinct repeated perceptual-hash groups:",
    perceptual_groups["perceptual_hash_dhash"].nunique()
)

display(perceptual_groups.head(20))

=== EXACT DUPLICATE SUMMARY ===
Records with repeated exact hashes: 5
Distinct repeated exact-hash groups: 1


,image_id,canonical_product_id,content_hash_sha256
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,fde2fa682cfa73aacdeca176a4a5e974df1e27ff41e144...



=== PERCEPTUAL DUPLICATE SUMMARY ===
Records with repeated perceptual hashes: 7
Distinct repeated perceptual-hash groups: 2


,image_id,canonical_product_id,perceptual_hash_dhash
32,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,0c4d4d4d4d4d4d4d
91,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,0c4d4d4d4d4d4d4d
238,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,d86b23031741338e
243,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,d86b23031741338e
245,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,d86b23031741338e
306,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,d86b23031741338e
402,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,d86b23031741338e


In [31]:
print("=== INVESTIGATE EXACT DUPLICATE GROUP ===")

duplicate_details = exact_groups.merge(
    interim_df[
        [
            "image_id",
            "title",
            "brand",
            "category",
            "image_url",
            "local_path",
            "source_provenance",
        ]
    ],
    on="image_id",
    how="left",
    validate="one_to_one",
)

display(
    duplicate_details[
        [
            "image_id",
            "canonical_product_id",
            "title",
            "brand",
            "category",
            "image_url",
            "local_path",
            "source_provenance",
        ]
    ].reset_index(drop=True)
)

print("\n=== UNIQUE SOURCE URLS ===")
print("Unique image URLs:", duplicate_details["image_url"].nunique())
print("\n=== UNIQUE PRODUCT TITLES ===")
print("Unique titles:", duplicate_details["title"].nunique())

=== INVESTIGATE EXACT DUPLICATE GROUP ===


,image_id,canonical_product_id,title,brand,category,image_url,local_path,source_provenance
0,amazon_B072MDJ9N2_img_0,amazon_B072MDJ9N2,Multi Focus Progressive Reading Glasses 3 Powe...,PASTL,Clothing & Fashion,https://m.media-amazon.com/images/I/01RmK+J4pJ...,data/raw/images/cache\B072MDJ9N2_img_0.jpg,Amazon Reviews'23 (Clothing & Fashion)
1,amazon_B08H1Y77B2_img_0,amazon_B08H1Y77B2,Amikadom Christmas Women Dress,Amikadom,Clothing & Fashion,https://m.media-amazon.com/images/I/01RmK+J4pJ...,data/raw/images/cache\B08H1Y77B2_img_0.jpg,Amazon Reviews'23 (Clothing & Fashion)
2,amazon_B089QN5VWB_img_0,amazon_B089QN5VWB,ZESICA Women's Long Sleeve Crew Neck Camouflag...,ZESICA,Clothing & Fashion,https://m.media-amazon.com/images/I/01RmK+J4pJ...,data/raw/images/cache\B089QN5VWB_img_0.jpg,Amazon Reviews'23 (Clothing & Fashion)
3,amazon_B01BXV8HH2_img_0,amazon_B01BXV8HH2,Luxurious Collections 100% Egyptian Cotton 600...,8,Home & Kitchen,https://m.media-amazon.com/images/I/01RmK+J4pJ...,data/raw/images/cache\B01BXV8HH2_img_0.jpg,Amazon Reviews'23 (Home & Kitchen)
4,amazon_B0823BQJ5H_img_0,amazon_B0823BQJ5H,Adults 1000 Pieces Animal Paradise Puzzle 1000...,Daliyfu,Toys & Games,https://m.media-amazon.com/images/I/01RmK+J4pJ...,data/raw/images/cache\B0823BQJ5H_img_0.jpg,Amazon Reviews'23 (Toys & Games)



=== UNIQUE SOURCE URLS ===
Unique image URLs: 1

=== UNIQUE PRODUCT TITLES ===
Unique titles: 5


In [32]:
print("=== SOURCE URL DUPLICATE INVESTIGATION ===")

url_col = "image_url"

url_counts = (
    interim_df.dropna(subset=[url_col])
    .groupby(url_col)
    .agg(
        image_count=("image_id", "nunique"),
        product_count=("canonical_product_id", "nunique"),
        titles=("title", lambda s: list(s.dropna().unique())),
        categories=("category", lambda s: list(s.dropna().unique())),
        brands=("brand", lambda s: list(s.dropna().unique())),
    )
    .sort_values("product_count", ascending=False)
)

display(url_counts[url_counts["product_count"] > 1].head(10))

print("\n=== EXACT HASH GROUP CHECK ===")
print("Repeated exact-hash records:", len(exact_groups))
print("Distinct product IDs:", exact_groups["canonical_product_id"].nunique())
print("Distinct source URLs:", duplicate_details["image_url"].nunique())

=== SOURCE URL DUPLICATE INVESTIGATION ===


,image_count,product_count,titles,categories,brands
image_url,,,,,
https://m.media-amazon.com/images/I/01RmK+J4pJL._AC_.gif,5,5,[Multi Focus Progressive Reading Glasses 3 Pow...,"[Clothing & Fashion, Home & Kitchen, Toys & Ga...","[PASTL, Amikadom, ZESICA, 8, Daliyfu]"



=== EXACT HASH GROUP CHECK ===
Repeated exact-hash records: 5
Distinct product IDs: 5
Distinct source URLs: 1


In [33]:
print("=== REUSED IMAGE URLS ===")

url_analysis = (
    interim_df.dropna(subset=["image_url"])
    .groupby("image_url")
    .agg(
        image_count=("image_id", "nunique"),
        product_count=("canonical_product_id", "nunique"),
        titles=("title", lambda s: list(s.dropna().unique())),
        categories=("category", lambda s: list(s.dropna().unique())),
        brands=("brand", lambda s: list(s.dropna().unique())),
    )
    .reset_index()
)

reused_urls = (
    url_analysis[url_analysis["product_count"] > 1]
    .sort_values("product_count", ascending=False)
)

print("Reused URL groups:", len(reused_urls))
display(reused_urls.head(10))

=== REUSED IMAGE URLS ===
Reused URL groups: 1


,image_url,image_count,product_count,titles,categories,brands
0,https://m.media-amazon.com/images/I/01RmK+J4pJ...,5,5,[Multi Focus Progressive Reading Glasses 3 Pow...,"[Clothing & Fashion, Home & Kitchen, Toys & Ga...","[PASTL, Amikadom, ZESICA, 8, Daliyfu]"


In [34]:
print("=== SECOND PERCEPTUAL-HASH GROUP ===")

second_hash = "0c4d4d4d4d4d4d4d"

second_group = canonical_df.loc[
    canonical_df["perceptual_hash_dhash"].eq(second_hash)
].copy()

display(
    second_group[
        [
            "image_id",
            "canonical_product_id",
            "marketplace",
            "image_url",
            "local_path",
            "content_hash_sha256",
            "perceptual_hash_dhash",
        ]
    ]
)

print("Records:", len(second_group))
print(
    "Unique exact hashes:",
    second_group["content_hash_sha256"].nunique()
)

=== SECOND PERCEPTUAL-HASH GROUP ===


,image_id,canonical_product_id,marketplace,image_url,local_path,content_hash_sha256,perceptual_hash_dhash
32,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,amazon,https://m.media-amazon.com/images/I/31nL8SOzvP...,data/raw/images/cache\B07QMZS32S_img_0.jpg,32cc764a4e7fb77623449c7c5161062790d598a29a9c74...,0c4d4d4d4d4d4d4d
91,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,amazon,https://m.media-amazon.com/images/I/21AwrfzIsg...,data/raw/images/cache\B0BWZX4TQ1_img_0.jpg,a6c43f8a52ef657b14fb0a6f3b08491288c7b125a2bf8f...,0c4d4d4d4d4d4d4d


Records: 2
Unique exact hashes: 2


In [36]:
print("=== NEAR-DUPLICATE ANALYSIS ===")

print("Candidate pairs:", len(near_duplicate_pairs))

if near_duplicate_pairs:
    display(near_duplicate_pairs[:20])
else:
    print("No near-duplicate candidate pairs found.")

=== NEAR-DUPLICATE ANALYSIS ===
Candidate pairs: 5


[{'image_id_1': 'amazon_B07QMZS32S_img_0',
  'product_id_1': 'amazon_B07QMZS32S',
  'image_id_2': 'amazon_B01LR80A6M_img_0',
  'product_id_2': 'amazon_B01LR80A6M',
  'hamming_distance': 2,
  'same_product': False},
 {'image_id_1': 'amazon_B07QMZS32S_img_0',
  'product_id_1': 'amazon_B07QMZS32S',
  'image_id_2': 'amazon_B00EP2ZX7Y_img_0',
  'product_id_2': 'amazon_B00EP2ZX7Y',
  'hamming_distance': 1,
  'same_product': False},
 {'image_id_1': 'amazon_B00XYMILHU_img_0',
  'product_id_1': 'amazon_B00XYMILHU',
  'image_id_2': 'amazon_B00EP2ZX7Y_img_0',
  'product_id_2': 'amazon_B00EP2ZX7Y',
  'hamming_distance': 2,
  'same_product': False},
 {'image_id_1': 'amazon_B01LR80A6M_img_0',
  'product_id_1': 'amazon_B01LR80A6M',
  'image_id_2': 'amazon_B0BWZX4TQ1_img_0',
  'product_id_2': 'amazon_B0BWZX4TQ1',
  'hamming_distance': 2,
  'same_product': False},
 {'image_id_1': 'amazon_B00EP2ZX7Y_img_0',
  'product_id_1': 'amazon_B00EP2ZX7Y',
  'image_id_2': 'amazon_B0BWZX4TQ1_img_0',
  'product_id_2

## Computer Vision — Preliminary Findings

### Dataset coverage
- Canonical image manifest: 3,909 records.
- Downloaded primary images: 520.
- All 520 downloaded image IDs match existing canonical records.
- Product ID and image URL mismatches between the canonical dataset and interim manifest: zero.

### Duplicate-image investigation
- Five records share one exact SHA-256 hash.
- The five records have different product IDs and titles, span multiple product categories, and share the same source image URL.
- Two perceptual-hash groups were identified, containing seven records in total.
- Repeated perceptual hashes indicate candidate visual matches, not confirmed fraud.

### Packaging consistency
- All 520 downloaded images have a numeric packaging-consistency score within the expected 0–1 range.
- A substantial number of images receive a neutral score of 0.50.
- Neutral scores must be interpreted in the context of OCR availability and the scoring implementation.

### Limitations and next actions
- The analysis currently covers 520 downloaded primary images, not all 3,909 image records.
- Investigate reused image URLs and validate candidate near-duplicates.
- Review OCR coverage and packaging-score semantics before using these features in the risk engine.
- Keep image-source data-quality anomalies separate from confirmed fraud labels.

In [37]:
near_duplicates_df = pd.DataFrame(near_duplicate_pairs)

print("=== NEAR-DUPLICATE CANDIDATES ===")
display(
    near_duplicates_df.sort_values(
        "hamming_distance"
    ).reset_index(drop=True)
)

print("\n=== DISTANCE DISTRIBUTION ===")
display(
    near_duplicates_df["hamming_distance"]
    .value_counts()
    .sort_index()
    .rename_axis("hamming_distance")
    .to_frame("candidate_pairs")
)

print("\n=== SAME-PRODUCT CHECK ===")
print(
    "Pairs from different products:",
    int((~near_duplicates_df["same_product"]).sum())
)

=== NEAR-DUPLICATE CANDIDATES ===


,image_id_1,product_id_1,image_id_2,product_id_2,hamming_distance,same_product
0,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,1,False
1,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,1,False
2,amazon_B00XYMILHU_img_0,amazon_B00XYMILHU,amazon_B00EP2ZX7Y_img_0,amazon_B00EP2ZX7Y,2,False
3,amazon_B07QMZS32S_img_0,amazon_B07QMZS32S,amazon_B01LR80A6M_img_0,amazon_B01LR80A6M,2,False
4,amazon_B01LR80A6M_img_0,amazon_B01LR80A6M,amazon_B0BWZX4TQ1_img_0,amazon_B0BWZX4TQ1,2,False



=== DISTANCE DISTRIBUTION ===


,candidate_pairs
hamming_distance,
1,2
2,3



=== SAME-PRODUCT CHECK ===
Pairs from different products: 5


In [38]:
print("=== NEAR-DUPLICATE IMAGE FILES ===")

image_lookup = (
    canonical_df[
        ["image_id", "canonical_product_id", "local_path"]
    ]
    .drop_duplicates("image_id")
    .set_index("image_id")
)

for _, pair in near_duplicates_df.iterrows():
    print(
        f"\nPair: {pair['image_id_1']} <-> {pair['image_id_2']}"
    )

    for image_id in [pair["image_id_1"], pair["image_id_2"]]:
        if image_id in image_lookup.index:
            path = image_lookup.loc[image_id, "local_path"]
            print(f"  {image_id}: {path}")
        else:
            print(f"  {image_id}: not found in canonical image records")

=== NEAR-DUPLICATE IMAGE FILES ===

Pair: amazon_B07QMZS32S_img_0 <-> amazon_B01LR80A6M_img_0
  amazon_B07QMZS32S_img_0: data/raw/images/cache\B07QMZS32S_img_0.jpg
  amazon_B01LR80A6M_img_0: data/raw/images/cache\B01LR80A6M_img_0.jpg

Pair: amazon_B07QMZS32S_img_0 <-> amazon_B00EP2ZX7Y_img_0
  amazon_B07QMZS32S_img_0: data/raw/images/cache\B07QMZS32S_img_0.jpg
  amazon_B00EP2ZX7Y_img_0: data/raw/images/cache\B00EP2ZX7Y_img_0.jpg

Pair: amazon_B00XYMILHU_img_0 <-> amazon_B00EP2ZX7Y_img_0
  amazon_B00XYMILHU_img_0: data/raw/images/cache\B00XYMILHU_img_0.jpg
  amazon_B00EP2ZX7Y_img_0: data/raw/images/cache\B00EP2ZX7Y_img_0.jpg

Pair: amazon_B01LR80A6M_img_0 <-> amazon_B0BWZX4TQ1_img_0
  amazon_B01LR80A6M_img_0: data/raw/images/cache\B01LR80A6M_img_0.jpg
  amazon_B0BWZX4TQ1_img_0: data/raw/images/cache\B0BWZX4TQ1_img_0.jpg

Pair: amazon_B00EP2ZX7Y_img_0 <-> amazon_B0BWZX4TQ1_img_0
  amazon_B00EP2ZX7Y_img_0: data/raw/images/cache\B00EP2ZX7Y_img_0.jpg
  amazon_B0BWZX4TQ1_img_0: data/raw/imag

## Final Computer Vision Analysis — Current Findings

### 1. Dataset coverage
- Canonical image records: 3,909.
- Downloaded primary images: 520.
- Products represented by downloaded images: 520, subject to verification of unique product counts.
- All downloaded image IDs match canonical records.
- No product ID or image URL mismatches were found between the canonical dataset and interim manifest.

### 2. Exact duplicate detection
- Five records share one repeated SHA-256 hash.
- The five records have different product IDs and titles.
- They share one image URL across multiple product categories.
- This is a potential image-source or dataset-mapping anomaly that requires investigation, not a confirmed fraud finding.

### 3. Perceptual similarity
- Two repeated perceptual-hash groups contain seven records in total.
- Five near-duplicate candidate pairs were identified.
- Candidate Hamming distances range from 1 to 2.
- All five candidate pairs involve different product IDs.
- Visual inspection is needed to validate these candidate matches.

### 4. OCR and packaging consistency
- OCR text is available for 204 of the 520 downloaded images.
- All packaging-consistency scores are within the expected 0–1 range.
- A substantial number of records receive the neutral score 0.50.
- The score's meaning must be validated against OCR availability and the scoring implementation.

### 5. Limitations
- Current image-analysis coverage is limited to the 520 downloaded primary images.
- The remaining canonical image records have not been validated through the same downloaded-image workflow.
- Hash similarity alone cannot establish product fraud or seller intent.
- Packaging-consistency scores should not be treated as calibrated fraud probabilities.

### 6. Next actions
- Visually inspect duplicate candidates.
- Investigate reused source image URLs.
- Validate the packaging-consistency implementation.
- Integrate validated image signals into the multimodal risk engine with explicit evidence and coverage indicators.